In [1]:
# pip install "lerobot[smolvla]"
# !source venv/bin/activate
# !pip install "lerobot[smolvla]"
# !pip install "lerobot[dataset]"
# !pip install --upgrade mistral-common


In [ ]:
import torch

from lerobot.datasets.lerobot_dataset import LeRobotDataset
from lerobot.policies.factory import make_pre_post_processors

# Swap this import per-policy
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy

: 

In [ ]:
# load a policy
model_id = "lerobot/smolvla_base"  # <- swap checkpoint
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

policy = SmolVLAPolicy.from_pretrained(model_id).to(device).eval()

print("\nInput shape:")
for key, feat in policy.config.input_features.items():
    print(f" - {key}: shape {feat.shape}")


preprocess, postprocess = make_pre_post_processors(
    policy.config,
    model_id,
    preprocessor_overrides={"device_processor": {"device": str(device)}},
)
# load a lerobotdataset
dataset = LeRobotDataset("lerobot/libero")

# pick an episode
episode_index = 0

# each episode corresponds to a contiguous range of frame indices
from_idx = dataset.meta.episodes["dataset_from_index"][episode_index]
to_idx   = dataset.meta.episodes["dataset_to_index"][episode_index]

# get a single frame from that episode (e.g. the first frame)
frame_index = from_idx
frame = dict(dataset[frame_index])

# 1. Remap LIBERO's camera keys to match smolvla_base's expected input_features
frame["observation.images.camera1"] = frame.pop("observation.images.image")
frame["observation.images.camera2"] = frame.pop("observation.images.image2")

# 2. Preprocess and pass `batch` (not `frame`) to select_action
batch = preprocess(frame)
with torch.inference_mode():
    pred_action = policy.select_action(batch)
    pred_action = postprocess(pred_action)